In [2]:
import pandas as pd
import numpy as np

from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise import accuracy

from surprise.model_selection import train_test_split

In [3]:
ratings = pd.read_csv(
    "../data/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [3]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

In [4]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [5]:
model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

In [6]:
model.fit(trainset)

In [7]:
predictions = model.test(testset)

In [8]:
rmse = accuracy.rmse(predictions)

print("RMSE:", rmse)

RMSE: 0.8729
RMSE: 0.8728931555711225


In [11]:
predictions[0]

Prediction(uid=1841, iid=3717, r_ui=1.0, est=2.2449976403111234, details={'was_impossible': False})

In [12]:
user_id = 1

watched_movies = ratings[
    ratings.userId == user_id
]["movieId"].tolist()

In [13]:
all_movies = ratings["movieId"].unique()

unwatched_movies = [
    movie
    for movie in all_movies
    if movie not in watched_movies
]

In [14]:
movie_scores = []

for movie_id in unwatched_movies:

    pred = model.predict(user_id, movie_id)

    movie_scores.append(
        (movie_id, pred.est)
    )

In [15]:
top_movies = sorted(
    movie_scores,
    key=lambda x: x[1],
    reverse=True
)[:10]

top_movies

[(110, 4.918221018688698),
 (3022, 4.889700661536264),
 (3421, 4.885042796635496),
 (318, 4.874445215705807),
 (2019, 4.8594102188560155),
 (1204, 4.831213292512891),
 (858, 4.824610352596962),
 (1276, 4.821943607072717),
 (922, 4.818127405767864),
 (904, 4.817951976110158)]

In [17]:
movies = pd.read_csv(
    "../data/movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)

In [18]:
recommended_ids = [movie for movie, _ in top_movies]

recommended_movies = movies[
    movies.movieId.isin(recommended_ids)
]

recommended_movies

,movieId,title,genres
108,110,Braveheart (1995),Action|Drama|War
315,318,"Shawshank Redemption, The (1994)",Drama
847,858,"Godfather, The (1972)",Action|Crime|Drama
892,904,Rear Window (1954),Mystery|Thriller
910,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),Film-Noir
1186,1204,Lawrence of Arabia (1962),Adventure|War
1256,1276,Cool Hand Luke (1967),Comedy|Drama
1950,2019,Seven Samurai (The Magnificent Seven) (Shichin...,Action|Drama
2953,3022,"General, The (1927)",Comedy
3352,3421,Animal House (1978),Comedy


In [19]:
def recommend_svd(
    user_id,
    model,
    ratings,
    movies,
    k=10
):

    watched_movies = ratings[
        ratings.userId == user_id
    ]["movieId"].tolist()

    all_movies = ratings["movieId"].unique()

    unwatched_movies = [
        movie
        for movie in all_movies
        if movie not in watched_movies
    ]

    scores = []

    for movie_id in unwatched_movies:

        pred = model.predict(user_id, movie_id)

        scores.append(
            (movie_id, pred.est)
        )

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )[:k]

    recommended_ids = [
        movie_id
        for movie_id, _ in scores
    ]

    return movies[
        movies.movieId.isin(recommended_ids)
    ]

In [20]:
recommend_svd(
    user_id=1,
    model=model,
    ratings=ratings,
    movies=movies,
    k=10
)

,movieId,title,genres
108,110,Braveheart (1995),Action|Drama|War
315,318,"Shawshank Redemption, The (1994)",Drama
847,858,"Godfather, The (1972)",Action|Crime|Drama
892,904,Rear Window (1954),Mystery|Thriller
910,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),Film-Noir
1186,1204,Lawrence of Arabia (1962),Adventure|War
1256,1276,Cool Hand Luke (1967),Comedy|Drama
1950,2019,Seven Samurai (The Magnificent Seven) (Shichin...,Action|Drama
2953,3022,"General, The (1927)",Comedy
3352,3421,Animal House (1978),Comedy


In [21]:
from collections import defaultdict
import numpy as np

In [22]:
def get_top_k(predictions, k=10):

    top_k = defaultdict(list)

    for pred in predictions:

        uid = pred.uid
        iid = pred.iid
        est = pred.est

        top_k[uid].append((iid, est))

    for uid, user_ratings in top_k.items():

        user_ratings.sort(
            key=lambda x: x[1],
            reverse=True
        )

        top_k[uid] = user_ratings[:k]

    return top_k

In [23]:
def precision_at_k(
    top_k,
    testset,
    threshold=4
):

    precisions = []

    for uid, user_ratings in top_k.items():

        recommended = [
            iid for iid, _ in user_ratings
        ]

        relevant = [
            iid
            for (u, iid, true_r) in testset
            if u == uid and true_r >= threshold
        ]

        if len(recommended) == 0:
            continue

        hits = len(
            set(recommended) & set(relevant)
        )

        precisions.append(
            hits / len(recommended)
        )

    return np.mean(precisions)

In [24]:
def recall_at_k(
    top_k,
    testset,
    threshold=4
):

    recalls = []

    for uid, user_ratings in top_k.items():

        recommended = [
            iid for iid, _ in user_ratings
        ]

        relevant = [
            iid
            for (u, iid, true_r) in testset
            if u == uid and true_r >= threshold
        ]

        if len(relevant) == 0:
            continue

        hits = len(
            set(recommended) & set(relevant)
        )

        recalls.append(
            hits / len(relevant)
        )

    return np.mean(recalls)

In [25]:
def ndcg_at_k(
    top_k,
    testset,
    threshold=4
):

    ndcgs = []

    for uid, user_ratings in top_k.items():

        recommended = [
            iid for iid, _ in user_ratings
        ]

        relevant = set(
            iid
            for (u, iid, true_r) in testset
            if u == uid and true_r >= threshold
        )

        dcg = 0

        for i, iid in enumerate(recommended):

            if iid in relevant:
                dcg += 1 / np.log2(i + 2)

        idcg = sum(
            1 / np.log2(i + 2)
            for i in range(
                min(len(relevant), len(recommended))
            )
        )

        if idcg == 0:
            continue

        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)

In [26]:
predictions = model.test(testset)

In [27]:
top_k = get_top_k(
    predictions,
    k=10
)

In [28]:
precision = precision_at_k(
    top_k,
    testset
)

recall = recall_at_k(
    top_k,
    testset
)

ndcg = ndcg_at_k(
    top_k,
    testset
)

print("Precision@10:", precision)
print("Recall@10:", recall)
print("NDCG@10:", ndcg)

Precision@10: 0.7538339397862023
Recall@10: 0.6406188961025355
NDCG@10: 0.8728692421360085
